## 0. Configuration et imports

In [ ]:
import os, sys, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
sys.path.insert(0, os.path.abspath('..'))
load_dotenv(find_dotenv(), override=True)

In [ ]:
# Modèle pour la génération des réponses du RAG
RAG_MODEL  = os.getenv("OPENROUTER_MODEL", "nvidia/nemotron-3-super-120b-a12b:free")

# Modèle pour l'évaluation des réponses du RAG (LLM-as-judge)
EVAL_MODEL = "openai/gpt-oss-20b:free"

print(f"RAG_MODEL  : {RAG_MODEL}")
print(f"EVAL_MODEL : {EVAL_MODEL}")

In [ ]:
from opensearchpy import OpenSearch
import pandas as pd
import os

# Client OpenSearch
os_client = OpenSearch(
    hosts=[{"host": os.getenv("OPENSEARCH_HOST", "localhost"),
            "port": int(os.getenv("OPENSEARCH_PORT", 9200))}],
    http_auth=(
        os.getenv("OPENSEARCH_USER", "admin"),
        os.getenv("OPENSEARCH_PASSWORD", "R@gTime2026#Store")
    ),
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
)
info = os_client.info()
print(f"OpenSearch {info['version']['number']} connecté")

# Chargement dataset prétraité
PREPROCESSED_PATH = "../data/processed/tickets_preprocessed.csv"
INDEX_NAME = "logistore_tickets"

if os.path.exists(PREPROCESSED_PATH):
    df_clean = pd.read_csv(PREPROCESSED_PATH)
    print(f"Dataset chargé — {df_clean.shape[0]} tickets, {df_clean.shape[1]} colonnes")
    print(f"   Colonnes : {list(df_clean.columns)}")
else:
    # Fallback sur le dataset brut si le dataset prétraité n'existe pas encore
    df_clean = pd.read_csv("../data/raw/dataset-tickets-multi-lang3-4k.csv")
    print(f" Prétraité non trouvé — dataset brut chargé ({df_clean.shape[0]} tickets)")

# Vérification des index OpenSearch
count = os_client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' — {count['count']} documents indexés")

display(df_clean.head(3))

## 1. Pré-traitement de l'ensemble de test

In [ ]:
import time, re, pandas as pd
from src.llm.llm_client import call_llm

EVAL_MODEL = "deepseek/deepseek-v4-flash:free"

QUERY_GEN_PROMPT = """\
Ticket de support client :
Sujet : {subject}
Problème : {body}

Écris UNE question courte (max 15 mots) en {lang} qu'un client aurait posée.
Réponds UNIQUEMENT avec la question. Aucune introduction, aucune explication, aucun guillemet."""

LANG_NAMES = {"en": "English", "fr": "français", "de": "Deutsch", "es": "español", "pt": "português"}

def generate_query(row):
    prompt = QUERY_GEN_PROMPT.format(
        subject=str(row["subject"])[:200],
        body=str(row["body"])[:400],
        lang=LANG_NAMES.get(row["language"], row["language"])
    )
    for attempt in range(3):
        try:
            raw = call_llm(prompt, model=EVAL_MODEL).strip()
            # Nettoyage : enlève guillemets, "Question :", numéros
            raw = re.sub(r'^["\'«»]|["\'»]$', '', raw).strip()
            raw = re.sub(r'^(Question\s*:|Q\s*:|\d+[\.\)])\s*', '', raw, flags=re.IGNORECASE).strip()
            # Validation : entre 4 et 20 mots, pas un prompt retourné
            words = raw.split()
            if 4 <= len(words) <= 25 and not re.match(
                r'(?i)^(we need|generate|nous devons|générer|produce|output|write a|create a|task:|instructions?:)',
                raw
            ):
                return raw
        except Exception as e:
            print(f"Erreur attempt {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return None  # échec après 3 tentatives

# Chargement du fichier source pour récupérer les tickets originaux
df_source = pd.read_csv("../data/raw/dataset-tickets-multi-lang3-4k.csv")

# Sélection de 10 tickets par langue (même seed que l'original)
df_sample = (
    df_source.groupby("language", group_keys=False)
    .apply(lambda g: g.sample(10, random_state=42))
    .reset_index(drop=True)
)

print(f"Tickets sélectionnés : {len(df_sample)}")
print(df_sample["language"].value_counts())

In [ ]:
# Chargement du pipeline

from src.retrieval.hybrid_search import hybrid_search
from src.llm.llm_client import rag_answer, call_llm
from src.llm.rag_pipeline import run_rag, to_rag_tickets

In [ ]:
# Configuration pour l'évaluation

EVAL_MODEL = os.getenv("OPENROUTER_MODEL")  # nvidia/nemotron
N_SAMPLES_PER_LANG = 10   # 10 tickets échantillonnés par langue, donc 50 tickets au total
LANGUAGES = ["en", "de", "fr", "es", "pt"]
TOP_K = 5
EVAL_OUTPUT = Path("../data/eval/")
EVAL_OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. Génération de l'ensemble de test et des requêtes

### 2.1. Ensemble de test

In [ ]:
def generate_query(row):
    prompt = QUERY_GEN_PROMPT.format(
        subject=str(row["subject"])[:200],
        body=str(row["body"])[:400],
        lang=LANG_NAMES.get(row["language"], row["language"])
    )
    # call_llm attend une liste de messages, pas une string
    messages = [{"role": "user", "content": prompt}]
    
    for attempt in range(3):
        try:
            raw = call_llm(messages, model=EVAL_MODEL).strip()
            # Nettoyage
            raw = re.sub(r'^["\'«»]|["\'»]$', '', raw).strip()
            raw = re.sub(r'^(Question\s*:|Q\s*:|\d+[\.\)])\s*', '', raw, flags=re.IGNORECASE).strip()
            # Validation : entre 4 et 25 mots, pas un prompt retourné
            words = raw.split()
            if 4 <= len(words) <= 25 and not re.match(
                r'(?i)^(we need|generate|nous devons|générer|produce|output|write a|create a|task:|instructions?:)',
                raw
            ):
                return raw
            else:
                print(f"Rejeté ({len(words)} mots): {raw[:80]}")
        except Exception as e:
            print(f"  Erreur attempt {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return None

In [ ]:
# Test rapide avant la boucle complète
test = generate_query(df_sample.iloc[0])
print("Test:", test)

In [ ]:
# Génération

queries = []
for i, row in df_sample.iterrows():
    q = generate_query(row)
    queries.append(q)
    status = "✅" if q else "❌"
    print(f"[{i+1:02d}/{len(df_sample)}] {row['language']} | {status} {q or 'ÉCHEC'}")
    time.sleep(0.3)  # rate limit

df_sample["query_generated"] = queries

# Suppression des échecs
df_eval_clean = df_sample[df_sample["query_generated"].notna()].reset_index(drop=True)

print(f"\n✅ Requêtes valides : {len(df_eval_clean)}/50")
print(df_eval_clean.groupby("language")["query_generated"].count())

In [ ]:
# Construction du test set avec les vrais IDs OpenSearch

# IDs depuis OpenSearch
resp = os_client.search(
    index=INDEX_NAME,
    body={
        "size": 5000,
        "query": {"match_all": {}},
        "_source": ["language", "type", "priority", "text"],
    }
)

# DataFrame avec les vrais IDs
os_records = []
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    text = src.get("text", "")
    parts = text.split(" | ", 2)
    os_records.append({
        "ticket_id": hit["_id"],
        "language":  src.get("language", ""),
        "type":      src.get("type", ""),
        "priority":  src.get("priority", ""),
        "subject":   parts[0].strip() if len(parts) > 0 else "",
        "body":      parts[1].strip()[:300] if len(parts) > 1 else text[:300],
    })

df_os = pd.DataFrame(os_records)
print(f"{len(df_os)} tickets récupérés depuis OpenSearch")

# Echantillonnage stratifié des tickets: 10 par langue
N_SAMPLES_PER_LANG = 10
LANGUAGES = ["en", "de", "fr", "es", "pt"]

df_sample = (
    df_os[df_os["language"].isin(LANGUAGES)]
    .groupby("language", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_SAMPLES_PER_LANG, len(g)), random_state=42))
    .reset_index(drop=True)
)

print(f"Echantillon : {len(df_sample)} tickets")
display(df_sample.groupby("language").size().rename("count").to_frame())
print("\nExemple IDs :", df_sample["ticket_id"].head(5).tolist())

### 2.2. Génération des requêtes

In [ ]:
generated_queries = []

for i, (_, row) in enumerate(df_sample.iterrows()):
    print(f"[{i+1:02d}/{len(df_sample)}] langue={row['language']} | ticket={row['ticket_id']}", end=" : ")

    query = generate_query(row)

    generated_queries.append({
        "ticket_id":       row["ticket_id"],
        "language":        row["language"],
        "type":            row.get("type", ""),
        "priority":        row.get("priority", ""),
        "subject":         row.get("subject", ""),
        "body":            row.get("body", ""),
        "query_generated": query,
    })

    print(f'"{query[:60]}..."' if query else "échec")

    # Sauvegarde intermédiaire tous les 10 tickets
    if (i + 1) % 10 == 0:
        pd.DataFrame(generated_queries).to_csv(
            "../data/eval/test_queries_partial.csv", index=False
        )
        print(f"Sauvegarde intermédiaire ({i+1} tickets)")

    time.sleep(0.5)

df_eval = pd.DataFrame(generated_queries)
n_failed = df_eval["query_generated"].isna().sum()
print(f"\nGénération terminée — {len(df_eval) - n_failed}/{len(df_eval)} requêtes générées ({n_failed} échecs)")

### 2.3. Sauvegarde des requêtes et aperçu par langue

In [ ]:
df_eval.to_csv("../data/eval/test_queries_clean.csv", index=False)
print("Sauvegardé → data/eval/test_queries_clean.csv\n")

for lang in LANGUAGES:
    print(f"─ {lang.upper()} ─")
    subset = df_eval[df_eval["language"] == lang]["query_generated"].tolist()
    for q in subset:
        print(f"  • {q}")
    print()

## 3. Métriques

### 3.1. Métriques d'évaluation du retrieval

In [ ]:
import numpy as np

def reciprocal_rank(retrieved_ids: list, relevant_id: str) -> float:
    """RR = 1/rang du premier doc pertinent, 0 si absent."""
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id == relevant_id:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved_ids: list, relevant_id: str, k: int = 5) -> float:
    """nDCG@K binaire — 1 seul doc pertinent (le ticket source)."""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k]):
        if doc_id == relevant_id:
            dcg = 1.0 / np.log2(i + 2)  # log2(rang+1), rang commence à 1
            break
    idcg = 1.0  # doc pertinent en position 1 = score max
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(retrieved_ids: list, relevant_id: str, k: int = 5) -> float:
    """Recall@K = 1 si le doc pertinent est dans top-K, 0 sinon."""
    return 1.0 if relevant_id in retrieved_ids[:k] else 0.0

def precision_at_k(retrieved_ids: list, relevant_id: str, k: int = 5) -> float:
    """Precision@K = nb pertinents dans top-K / K (binaire → 0 ou 1/K)."""
    hits = sum(1 for doc_id in retrieved_ids[:k] if doc_id == relevant_id)
    return hits / k

print("Fonctions métriques définies")

In [ ]:
On évalue ici.

In [ ]:
import time

# Chargement de l'ensemble de test
df_eval = pd.read_csv("../data/eval/test_queries_clean.csv")
df_eval = df_eval.dropna(subset=["query_generated"]).reset_index(drop=True)
print(f"Jeu de test chargé — {len(df_eval)} requêtes valides")

retrieval_records = []

for i, row in df_eval.iterrows():
    query      = row["query_generated"]
    lang       = row["language"]
    relevant_id = row["ticket_id"]

    try:
        results = hybrid_search(
            query=query,
            top_k=5,
            filters={"language": lang},
        )
        retrieved_ids = [r["_id"] for r in results]
    except Exception as e:
        print(f"Erreur retrieval [{i}] : {e}")
        retrieved_ids = []

    retrieval_records.append({
        "ticket_id":   relevant_id,
        "language":    lang,
        "type":        row.get("type", ""),
        "query":       query,
        "rr":          reciprocal_rank(retrieved_ids, relevant_id),
        "ndcg5":       ndcg_at_k(retrieved_ids, relevant_id, k=5),
        "recall5":     recall_at_k(retrieved_ids, relevant_id, k=5),
        "precision5":  precision_at_k(retrieved_ids, relevant_id, k=5),
        "retrieved_ids": str(retrieved_ids),
    })

    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(df_eval)}] MRR courant : {np.mean([r['rr'] for r in retrieval_records]):.3f}")

    time.sleep(0.2)

df_retrieval = pd.DataFrame(retrieval_records)
df_retrieval.to_csv("../data/eval/retrieval_results.csv", index=False)
print(f"\n✅ Évaluation terminée — {len(df_retrieval)} requêtes évaluées")

In [ ]:
metrics_global = {
    "MRR":         df_retrieval["rr"].mean(),
    "nDCG@5":      df_retrieval["ndcg5"].mean(),
    "Recall@5":    df_retrieval["recall5"].mean(),
    "Precision@5": df_retrieval["precision5"].mean(),
}

print("=" * 40)
print("Récapitulatif des métriques pour l'évaluation du retrieval")
print("=" * 40)
for k, v in metrics_global.items():
    bar = "█" * int(v * 20)
    print(f"  {k:<12} {v:.3f}  {bar}")
print("=" * 40)

display(pd.DataFrame([metrics_global]).round(3))

| Métrique            | Score   | Interprétation                                                  |
| ------------------- | ------- | --------------------------------------------------------------- |
| MRR = 0.312         | Moyen   | Le bon ticket apparaît en moyenne vers la position 3            |
| nDCG@5 = 0.383      | Correct | Le classement est partiellement bon                             |
| Recall@5 = 0.600    | Bon     | 60% des fois, le bon ticket est dans les top-5                  |
| Precision@5 = 0.120 | Normal  | 1 ticket pertinent sur 5, attendu avec 1 seul doc de référence  |

La Precision@5 à 0.12 est normale: avec 1 seul document pertinent par requête, le maximum théorique est 1/5 = 0.20. On est à 60% du maximum.

Le Recall@5 à 0.60 est le chiffre le plus important: le système trouve le bon ticket dans les top-5 dans 6 cas sur 10.

In [ ]:
df_by_lang = (
    df_retrieval
    .groupby("language")[["rr", "ndcg5", "recall5", "precision5"]]
    .mean()
    .round(3)
    .rename(columns={"rr": "MRR", "ndcg5": "nDCG@5",
                     "recall5": "Recall@5", "precision5": "Precision@5"})
)

display(df_by_lang)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
df_by_lang.plot(kind="bar", ax=ax, color=["#01696F","#20808D","#BCE2E7","#1B474D"])
ax.set_title("Métriques retrieval par langue", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: langue x type

import seaborn as sns

pivot = df_retrieval.pivot_table(
    values="recall5", index="language", columns="type", aggfunc="mean"
).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGn",
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title("Recall@5 par langue et type de ticket", fontweight="bold")
plt.tight_layout()
plt.show()

| Langue | Recall@5 | Interprétation                                                   |
| ------ | -------- | ---------------------------------------------------------------- |
| EN   | 0.80     | Excellent: le modèle MiniLM performe très bien en anglais       |
| FR     | 0.60     | Correct: dans la moyenne                                        |
| PT     | 0.60     | Correct: dans la moyenne                                        |
| DE     | 0.50     | Passable: la moitié des requêtes trouvent le bon ticket            |
| ES   | 0.50     | Passable: MRR très faible (0.145) : quand il trouve, c'est tard |

Conclusion sur les métriques du retrieval:

EN domine largement: MRR 0.603 vs 0.145-0.325 pour les autres. C'est ce qui est attendu, car paraphrase-multilingual-MiniLM-L12-v2 est entraîné majoritairement sur des données anglaises, même s'il supporte plus de 50 langues.

ES est le point faible: Recall@5 = 0.50 mais MRR = 0.145, ce qui signifie que quand le bon ticket est trouvé, il est très mal classé (position 5-7 en moyenne). Le BM25 souffre probablement du vocabulaire espagnol.

DE sous-performe malgré un corpus plus grand (848 tickets vs 476 FR); cela suggère que les requêtes générées par le LLM en allemand sont moins naturelles.

### 3.2. Métriques d'évaluation de la génération

In [ ]:
FAITHFULNESS_PROMPT = """Évalue si la réponse est fidèle au contexte fourni (pas d'informations inventées).

Contexte : {context}
Réponse : {answer}

Réponds sur UNE SEULE LIGNE avec exactement ce format JSON :
{{"score": X, "reason": "..."}}

Remplace X par un entier entre 0 et 5 :
0 = complètement inventé, 3 = partiellement fidèle, 5 = entièrement fidèle"""

RELEVANCY_PROMPT = """Évalue si la réponse répond directement à la question posée.

Question : {query}
Réponse : {answer}

Réponds sur UNE SEULE LIGNE avec exactement ce format JSON :
{{"score": X, "reason": "..."}}

Remplace X par un entier entre 0 et 5 :
0 = hors sujet, 3 = partiellement pertinent, 5 = répond parfaitement"""

CONTEXT_PRECISION_PROMPT = """Évalue si le contexte récupéré est pertinent pour répondre à la question.

Question : {query}
Contexte : {context}

Réponds sur UNE SEULE LIGNE avec exactement ce format JSON :
{{"score": X, "reason": "..."}}

Remplace X par un entier entre 0 et 5 :
0 = contexte inutile, 3 = partiellement pertinent, 5 = parfaitement pertinent"""

print("Prompts d'évaluation définis")

In [ ]:
import json
import re

def parse_llm_json(raw: str) -> dict:
    """Extrait le JSON d'une réponse LLM même avec du texte parasite."""
    # Cherche tous les blocs { ... } — sans échappement \{
    matches = re.findall(r'\{[^{}]*\}', raw, re.DOTALL)
    for match in matches:
        try:
            parsed = json.loads(match)
            if "score" in parsed:
                try:
                    parsed["score"] = int(parsed["score"])
                except (ValueError, TypeError):
                    pass
                # Rejette si reason est le placeholder d'exemple
                if parsed.get("reason", "") in ("explication courte", "..."):
                    continue
                return parsed
        except json.JSONDecodeError:
            continue

    # Fallback regex
    score_match  = re.search(r'"score"\s*:\s*(\d)', raw)
    reason_match = re.search(r'"reason"\s*:\s*"([^"]*)"', raw)
    if score_match:
        reason = reason_match.group(1) if reason_match else "extrait par regex"
        if reason not in ("explication courte", "..."):
            return {"score": int(score_match.group(1)), "reason": reason}

    return {"score": None, "reason": "JSON non parseable"}

print("parse_llm_json — regex sans échappement et rejet des placeholders")


def evaluate_rag_response(query: str, answer: str, context: str, model: str = None) -> dict:
    if model is None:
        model = EVAL_MODEL

    results = {}

    evaluations = [
        ("faithfulness", [
            {"role": "system", "content": "You are an evaluator. Reply ONLY with a single JSON object on one line. No text before or after."},
            {"role": "user", "content": f"""Does this answer contain only information from the context?

CONTEXT: {context[:1200]}

ANSWER: {answer[:300]}

Reply: {{"score": X, "reason": "brief explanation"}}
X: 0=hallucinated, 3=partially faithful, 5=fully faithful"""},
        ]),
        ("relevancy", [
            {"role": "system", "content": "You are an evaluator. Reply ONLY with a single JSON object on one line. No text before or after."},
            {"role": "user", "content": f"""Does this answer respond to the question?

QUESTION: {query}
ANSWER: {answer[:300]}

Reply: {{"score": X, "reason": "brief explanation"}}
X: 0=off-topic, 3=partially relevant, 5=perfectly answers the question"""},
        ]),
        ("context_precision", [
            {"role": "system", "content": "You are an evaluator. Reply ONLY with a single JSON object on one line. No text before or after."},
            {"role": "user", "content": f"""Is this context useful to answer the question?

QUESTION: {query}
CONTEXT: {context[:1200]}

Reply: {{"score": X, "reason": "brief explanation"}}
X: 0=useless, 3=partially useful, 5=perfectly relevant"""},
        ]),
    ]

    for metric_name, messages in evaluations:
        for attempt in range(4):  # 4 tentatives max
            try:
                raw = call_llm(messages, model=model, temperature=0.0, max_tokens=100)
                parsed = parse_llm_json(raw)
                results[metric_name] = parsed.get("score")
                results[f"{metric_name}_reason"] = parsed.get("reason", "")
                break  # succès → passe au prochain métrique
            except Exception as e:
                if "429" in str(e):
                    wait = 5 * (attempt + 1)  # 5s, 10s, 15s, 20s
                    print(f"  ⏳ 429 {metric_name} — attente {wait}s (attempt {attempt+1}/4)")
                    time.sleep(wait)
                else:
                    results[metric_name] = None
                    results[f"{metric_name}_reason"] = str(e)
                    break
        else:
            # 4 tentatives épuisées
            results[metric_name] = None
            results[f"{metric_name}_reason"] = "429 après 4 tentatives"

        time.sleep(2.0)  # pause systématique entre métriques

    return results

print("evaluate_rag_response; retry 429 avec backoff linéaire")

In [ ]:
scores = evaluate_rag_response(query, answer, context)
print(scores)

In [ ]:
# Liste des modèles disponibles

import httpx

with httpx.Client(timeout=15) as client:
    resp = client.get(
        "https://openrouter.ai/api/v1/models",
        headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"}
    )
    models = resp.json()["data"]
    free = [m for m in models if m.get("pricing", {}).get("prompt") == "0"]
    # Filtre les modèles "thinking" (mauvais pour JSON)
    non_thinking = [m for m in free if "thinking" not in m["id"] 
                    and "reasoning" not in m["id"]
                    and "nemotron" not in m["id"]]
    for m in non_thinking[:15]:
        print(m["id"])

In [ ]:
rag_result = rag_answer(
    query=query,
    tickets=tickets,
    lang=lang,
    model=RAG_MODEL,      # Nemotron pour la génération
    max_tokens=512,
)

scores = evaluate_rag_response(
    query, answer, context,
    model=EVAL_MODEL       # OpenAI pour l'évaluation
)

In [ ]:
Evaluation sur les 50 requêtes:

In [ ]:
rag_records = []

for i, row in df_eval.iterrows():
    query = row["query_generated"]
    lang  = row["language"]

    print(f"[{len(rag_records)+1:02d}/{len(df_eval)}] lang={lang} | {query[:50]}...")

    try:
        retrieval  = hybrid_search(query=query, top_k=5, filters={"language": lang})
        tickets    = to_rag_tickets(retrieval[:5])
        rag_result = rag_answer(
            query=query,
            tickets=tickets,
            lang=lang,
            model=RAG_MODEL,
            max_tokens=512,
        )
        answer  = rag_result["answer"]
        context = "\n\n".join(
            f"[{j+1}] {t['subject']} | {t['body_preview']}"
            for j, t in enumerate(tickets)
        )
        scores = evaluate_rag_response(query, answer, context)

    except Exception as e:
        print(f"  ⚠️  Erreur : {e}")
        answer = ""
        scores = {
            "faithfulness": None, "relevancy": None, "context_precision": None,
            "faithfulness_reason": "", "relevancy_reason": "", "context_precision_reason": "",
        }

    rag_records.append({
        "ticket_id": row["ticket_id"],
        "language":  lang,
        "query":     query,
        "answer":    answer[:300],
        **scores,
    })

    if len(rag_records) % 10 == 0:
        pd.DataFrame(rag_records).to_csv("../data/eval/rag_results_partial.csv", index=False)
        print(f"   💾 Sauvegarde intermédiaire ({len(rag_records)}/50)")

    time.sleep(1.0)

df_rag = pd.DataFrame(rag_records)
df_rag.to_csv("../data/eval/rag_results.csv", index=False)
print(f"\n✅ Évaluation RAG terminée — {len(df_rag)} requêtes évaluées")

In [ ]:
Résultats RAG

In [ ]:
# Scores moyens globaux (sur 5)
metrics_rag = {
    "Faithfulness":      df_rag["faithfulness"].dropna().mean(),
    "Answer Relevancy":  df_rag["relevancy"].dropna().mean(),
    "Context Precision": df_rag["context_precision"].dropna().mean(),
}

print("=" * 45)
print("MÉTRIQUES RAG — GLOBAL (score /5)")
print("=" * 45)
for k, v in metrics_rag.items():
    bar = "█" * int(v * 4)
    print(f"  {k:<20} {v:.2f}/5  {bar}")
print("=" * 45)

# Par langue
df_rag_lang = (
    df_rag
    .groupby("language")[["faithfulness", "relevancy", "context_precision"]]
    .mean()
    .round(2)
    .rename(columns={
        "faithfulness":     "Faithfulness",
        "relevancy":        "Relevancy",
        "context_precision":"Context Precision"
    })
)
display(df_rag_lang)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
df_rag_lang.plot(kind="bar", ax=ax, color=["#01696F", "#20808D", "#BCE2E7"])
ax.set_title("Métriques RAG par langue (score /5)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Score moyen /5")
ax.set_ylim(0, 5.5)
ax.axhline(y=2.5, color="gray", linestyle="--", linewidth=0.8, label="Seuil acceptable (2.5/5)")
ax.legend()
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Diagnostic valeurs manquantes ─────────────────────────────────────────
print("Valeurs manquantes par métrique :")
print(df_rag[["faithfulness", "relevancy", "context_precision"]].isna().sum())
print(f"\nTotal lignes : {len(df_rag)}")
print(f"\nDistribution faithfulness : {df_rag['faithfulness'].value_counts(dropna=False).to_dict()}")
print(f"Distribution relevancy    : {df_rag['relevancy'].value_counts(dropna=False).to_dict()}")

In [ ]:
Points forts :

    FR : Relevancy 4.25 et Context Precision 4.25 — le retrieval FR est excellent (confirmé par MRR=0.60)

    EN : Relevancy 3.67 — réponses pertinentes

Points faibles :

    Faithfulness globale à 2.11/5 — Nemotron a tendance à halluciner ou extrapoler au-delà du contexte fourni

    PT : scores très bas (1.25/5) — cohérent avec le faible MRR=0.275 en retrieval

    FR Faithfulness à 1.67 — paradoxe : bon retrieval mais réponses pas fidèles au contexte

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# 4.1 Tableau de synthèse global
print("=" * 55)
print("SYNTHÈSE FINALE — SYSTÈME RAG LOGISTORE")
print("=" * 55)
print(f"\n📊 RETRIEVAL (hybride BM25 + kNN, RRF k=60)")
print(f"   MRR          : {0.391:.3f}  (EN=0.453, FR=0.600)")
print(f"   nDCG@5       : {0.423:.3f}")
print(f"   Recall@5     : {0.520:.3f}")

faith  = df_rag["faithfulness"].dropna().mean()
relev  = df_rag["relevancy"].dropna().mean()
ctx    = df_rag["context_precision"].dropna().mean()

print(f"\n🤖 GÉNÉRATION RAG (Nemotron 120B, éval OpenAI V4 Flash)")
print(f"   Faithfulness      : {faith:.2f}/5")
print(f"   Answer Relevancy  : {relev:.2f}/5")
print(f"   Context Precision : {ctx:.2f}/5")
print("=" * 55)

In [ ]:
from matplotlib.path import Path
import matplotlib.patches as patches

categories = ["MRR\n(×5)", "nDCG@5\n(×5)", "Recall@5\n(×5)",
              "Faithfulness", "Relevancy", "Context\nPrecision"]
N = len(categories)

# Normalise retrieval sur 5 pour même échelle que RAG
values = [
    0.391 * 5,   # MRR × 5
    0.423 * 5,   # nDCG@5 × 5
    0.520 * 5,   # Recall@5 × 5
    faith,
    relev,
    ctx,
]
values_plot = values + [values[0]]  # ferme le polygone

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Fond
ax.set_facecolor("#F8F9FA")
fig.patch.set_facecolor("#F8F9FA")

# Grilles
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"], fontsize=8, color="gray")

# Seuil acceptable
threshold = [2.5] * (N + 1)
ax.plot(angles, threshold, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax.fill(angles, threshold, color="gray", alpha=0.05)

# Courbe système
ax.plot(angles, values_plot, color="#01696F", linewidth=2.5, linestyle="solid")
ax.fill(angles, values_plot, color="#01696F", alpha=0.25)

# Labels axes
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight="bold", color="#28251D")

# Valeurs sur les points
for angle, value, cat in zip(angles[:-1], values, categories):
    ax.annotate(
        f"{value:.2f}",
        xy=(angle, value),
        xytext=(angle, value + 0.35),
        fontsize=9,
        ha="center",
        color="#01696F",
        fontweight="bold",
    )

# Titre
ax.set_title(
    "Radar sur la Performance du système RAG\n(retrieval × 5 | génération /5)",
    fontsize=12, fontweight="bold", color="#28251D", pad=30
)

# Légende
legend_elements = [
    mpatches.Patch(facecolor="#01696F", alpha=0.4, label="Système RAG"),
    mpatches.Patch(facecolor="gray", alpha=0.15, label="Seuil acceptable (2.5/5)"),
]
ax.legend(handles=legend_elements, loc="upper right",
          bbox_to_anchor=(1.3, 1.15), fontsize=9)

plt.tight_layout()
plt.savefig("../data/eval/radar_rag.png", dpi=150, bbox_inches="tight", pad_inches=0.5)
plt.show()
print("✅ Radar sauvegardé → data/eval/radar_rag.png")

In [ ]:
print("""
RECOMMANDATIONS
───────────────────────────────────────────────────
1. Faithfulness faible (2.11/5)
   → Ajouter une instruction système "réponds UNIQUEMENT
     à partir du contexte fourni, sans ajouter d'informations"
   → Réduire max_tokens (512→256) pour limiter les extrapolations

2. PT sous-performant (MRR=0.275, Relevancy=1.25/5)
   → Vérifier la couverture du modèle d'embedding pour le portugais
   → Envisager un modèle spécialisé ou un fine-tuning

3. ES/DE retrieval moyen (MRR~0.275-0.35)
   → Augmenter le poids BM25 dans RRF pour ces langues
   → Ou ajouter un re-ranker multilingue (cross-encoder)

4. Context Precision correcte (3.15/5)
   → Le retrieval hybride ramène des tickets pertinents
   → Marge d'amélioration avec query expansion
───────────────────────────────────────────────────
""")